Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnx
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Load Data

MLP Model

In [3]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
graph = onnx_model.graph
#print(onnx_model)
weights = {}
for initializer in onnx_model.graph.initializer:
    weights[initializer.name] = onnx.numpy_helper.to_array(initializer)

#print(weights)

In [10]:
import random
index = 0
synthData = []
while index <= 100000:
    random_vector =  []
    for i in range(0,13):
        random_number = random.uniform(0, 1)
        random_vector.append(random_number)
    synthData.append(random_vector)
    index+=1


In [11]:
# SET UP DATA INPUT TO TRY TO FIND REAL INPUT DATA BASED ON OUTPUTS
testX = synthData
input_df = pd.DataFrame(testX, columns=["inputA","inputB", "inputC", "inputD", "inputE", "inputF", "inputG", "inputH", "inputI", "inputJ", "inputK", "inputL", "inputM"])
print(input_df.head())
knownOutput = -0.14729762 

     inputA    inputB    inputC    inputD    inputE    inputF    inputG  \
0  0.513120  0.580572  0.554522  0.258283  0.957459  0.934082  0.115877   
1  0.141507  0.705021  0.610590  0.265069  0.336565  0.059089  0.746642   
2  0.290318  0.301358  0.798742  0.982025  0.436784  0.590326  0.477800   
3  0.257090  0.508479  0.123970  0.036969  0.862480  0.608924  0.804389   
4  0.934422  0.464551  0.369485  0.142829  0.464838  0.661685  0.344177   

     inputH    inputI    inputJ    inputK    inputL    inputM  
0  0.799028  0.193424  0.743708  0.538897  0.017992  0.633734  
1  0.082508  0.147648  0.799130  0.092327  0.888050  0.852930  
2  0.838730  0.330649  0.540156  0.842716  0.769170  0.821642  
3  0.063369  0.529725  0.310491  0.443137  0.423146  0.352661  
4  0.404029  0.216566  0.894022  0.074992  0.360786  0.234872  


In [12]:
import onnxruntime as rt

## test prediction 
final_prediction = []

onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(input_df, dtype=np.float32) 
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
final_prediction = outputs[0]
output_df = pd.DataFrame(final_prediction, columns=["output"])
data_df = pd.concat([input_df, output_df], axis = 1)
data_df.head(5)


,inputA,inputB,inputC,inputD,inputE,inputF,inputG,inputH,inputI,inputJ,inputK,inputL,inputM,output
0,0.513120,0.580572,0.554522,0.258283,0.957459,0.934082,0.115877,0.799028,0.193424,0.743708,0.538897,0.017992,0.633734,0.445425
1,0.141507,0.705021,0.610590,0.265069,0.336565,0.059089,0.746642,0.082508,0.147648,0.799130,0.092327,0.888050,0.852930,0.065690
2,0.290318,0.301358,0.798742,0.982025,0.436784,0.590326,0.477800,0.838730,0.330649,0.540156,0.842716,0.769170,0.821642,0.277912
3,0.257090,0.508479,0.123970,0.036969,0.862480,0.608924,0.804389,0.063369,0.529725,0.310491,0.443137,0.423146,0.352661,0.256073
4,0.934422,0.464551,0.369485,0.142829,0.464838,0.661685,0.344177,0.404029,0.216566,0.894022,0.074992,0.360786,0.234872,0.841423


In [13]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


x = data_df["output"].to_numpy().reshape(-1,1)
y = data_df.drop(columns="output").to_numpy()
X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=42, train_size = .7)

invModel = MLPRegressor(hidden_layer_sizes=(500, 300), max_iter=5000, activation='relu', solver='adam', random_state=42)
# Train the model
invModel.fit(X_train, y_train)

y_pred = invModel.predict(X_test)
print(y_pred)
print(y_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae}")


[[0.09256721 0.5502292  0.49356696 ... 0.5098059  0.51921934 0.5365287 ]
 [0.6256572  0.49313164 0.5042653  ... 0.5014634  0.508274   0.51637644]
 [0.8943472  0.46670392 0.51644003 ... 0.5041635  0.5061113  0.49265942]
 ...
 [0.8592417  0.47925434 0.5111109  ... 0.50092787 0.5075371  0.50234765]
 [0.9281359  0.4216922  0.52470136 ... 0.5098355  0.4989987  0.45847216]
 [0.80516195 0.48576903 0.50781727 ... 0.49983126 0.5082206  0.50837153]]
[[0.08058958 0.87432652 0.03885636 ... 0.88121096 0.98604594 0.04020797]
 [0.55943905 0.63012601 0.72386353 ... 0.50383323 0.01214353 0.07612977]
 [0.81380243 0.34039842 0.40994827 ... 0.84321785 0.5765269  0.31404904]
 ...
 [0.80152977 0.27933637 0.93432755 ... 0.71034198 0.31567443 0.87238329]
 [0.99464775 0.9851485  0.49881506 ... 0.49997684 0.73545732 0.41473058]
 [0.81168618 0.69476256 0.28697516 ... 0.77055131 0.38303005 0.20487168]]
Mean Absolute Error (MAE): 0.23376475849584866


In [14]:
import onnx
import onnxruntime as ort
import numpy as np
from sklearn.neural_network import MLPRegressor

# Hidden layer parameters
output_weights = weights["coefficient"]
output_bias = weights["intercepts"]

# Output layer parameters
hidden_weights = weights["coefficient1"]
hidden_bias = weights["intercepts1"]


# The second dimension of the weight matrix is the number of neurons
hidden_layer_size = hidden_weights.shape[1]


In [15]:
# Instantiate model
mlp_model = MLPRegressor(
    hidden_layer_sizes=(hidden_layer_size,),
    activation='relu', 
    solver='adam',     
    max_iter=1,       
    random_state=42,
    warm_start=True,
)

dummy_input = np.zeros((1, ), dtype=np.float32).reshape(-1, 1)
dummy_output = np.zeros((1,13), dtype=np.float32)
mlp_model.partial_fit(dummy_input, dummy_output)

# Assign known weights and biases

mlp_model.coefs_[0] = hidden_bias
print(mlp_model.coefs_[0].shape)
mlp_model.intercepts_[0] = hidden_weights.T
print(mlp_model.intercepts_[0].shape)
mlp_model.coefs_[1] = output_bias.T
print(mlp_model.coefs_[1].shape)
mlp_model.intercepts_[1] = output_weights.T
print(mlp_model.intercepts_[1].shape)


(1, 1)
(1, 500)
(500, 1)
(500, 13)


In [17]:
#data = manualX.reshape(1, -1) 
#print(data)
inputs_a = data_df["output"].to_numpy().reshape(-1, 1) 
print(inputs_a)
knownOutput = [[-0.14729762]] #for testing?

mlp_model.predict(inputs_a) 


[[-0.13947955]
 [-0.52522373]
 [ 0.5434603 ]
 ...
 [ 0.44136378]
 [ 0.5637065 ]
 [-0.62273574]]


ValueError: non-broadcastable output operand with shape (100001,1) doesn't match the broadcast shape (100001,500)

In [11]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time']) 
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df) 

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [12]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

In [14]:
# Create test value
manualX = X_test
manualY = y_test[0]
print(manualX)

[[-0.09924674  0.48484848  0.93       ...  0.          0.
   1.        ]
 [-0.09924674  0.48484848  0.93       ...  0.          0.25881905
   0.96592583]
 [-0.09546565  0.48484848  0.93       ...  0.          0.5
   0.8660254 ]
 ...
 [-0.00225057  0.57373737  0.894      ...  1.         -0.70710678
   0.70710678]
 [-0.01743513  0.57373737  0.894      ...  1.         -0.5
   0.8660254 ]
 [-0.03103252  0.57373737  0.894      ...  1.         -0.25881905
   0.96592583]]


In [ ]:
inputsL = []
inputsL.append(manualX.tolist())
#print(inputsL)

import onnxruntime as rt
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(inputsL, dtype=np.float32) 
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)